In [1]:
from langchain_community.document_loaders import PDFPlumberLoader

loader = PDFPlumberLoader(r"..\data\Employee_Benefits_Guide_2026_v1.pdf")
documents = loader.load()

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)
texts = text_splitter.split_documents(documents)

In [4]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
load_dotenv(override=True)

embeddings = GoogleGenerativeAIEmbeddings(model="gemini-embedding-2-preview")

In [5]:
# uv add faiss-cpu
from langchain_community.vectorstores import FAISS

vectorstore = FAISS.from_documents(texts, embeddings)

In [6]:
vectorstore.save_local("./faiss_index")

In [7]:
vectorstore = FAISS.load_local(
    "./faiss_index",
    embeddings,
    allow_dangerous_deserialization=True # 데이터 역직렬화 허용
)

In [8]:
query = "결혼하면 얼마를 받을 수 있을까?"

In [9]:
results = vectorstore.similarity_search(query, k=3)

In [10]:
from langchain_core.prompts import PromptTemplate

prompt = PromptTemplate.from_template(
    """다음 컨텍스트만 사용해 질문에 답하세요.
컨텍스트: {context}

질문: {question}
"""
)

In [12]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()

In [13]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")

chain = prompt | model | parser

In [14]:
res = chain.invoke({"context": results, "question": query})

In [15]:
res

'결혼하면 본인의 경우 100만원의 경조금과 화환이 지원됩니다. 자녀 결혼 시에는 50만원, 형제/자매 결혼 시에는 30만원이 지원됩니다.'

---

In [16]:
from langchain.chat_models import init_chat_model

model = init_chat_model("google_genai:gemini-2.5-flash-lite")
model.invoke(query)

AIMessage(content='결혼하면 받는 금액은 나라, 지역, 그리고 개인의 상황에 따라 매우 다양하게 달라집니다. 일반적으로 결혼으로 인해 직접적으로 "받는" 금액은 법적으로 정해진 것은 없으며, 주로 다음과 같은 형태로 나타납니다.\n\n**1. 축의금 (결혼식 하객으로부터 받는 선물):**\n\n*   **가장 일반적인 형태:** 결혼식에 참석한 하객들이 신랑, 신부에게 축하의 의미로 전달하는 돈입니다.\n*   **금액 범위:**\n    *   **친한 친구/가족:** 5만원 ~ 10만원 이상 (관계에 따라 더 많을 수도 있습니다.)\n    *   **일반 지인:** 3만원 ~ 5만원\n    *   **직장 동료:** 3만원 ~ 5만원\n*   **영향 요인:**\n    *   **관계의 친밀도:** 당연히 가까운 사이일수록 더 많이 냅니다.\n    *   **식사 제공 여부:** 식사를 제공하는 경우, 보통 식사 비용을 고려하여 금액이 정해집니다.\n    *   **지역 및 문화:** 지역이나 문화권에 따라 축의금 문화나 평균 금액이 다를 수 있습니다.\n    *   **개인의 경제 상황:** 하객 본인의 경제적 여유에 따라 달라집니다.\n\n**2. 혼수/예물:**\n\n*   **전통적인 의미:** 신랑, 신부가 결혼 후 함께 살아갈 가정을 꾸리는 데 필요한 물품이나 돈을 서로 주고받는 것을 의미합니다.\n*   **형태:**\n    *   **현금:** 직접적으로 돈을 주고받는 경우입니다.\n    *   **가전제품, 가구, 자동차 등:** 생활에 필요한 물품을 선물하는 경우입니다.\n    *   **예물:** 결혼반지, 시계 등 특별한 의미를 담은 보석류를 주고받습니다.\n*   **금액/가치:** 이는 법적으로 정해진 것이 전혀 없으며, **전적으로 당사자 간의 합의와 경제적 능력에 따라 결정**됩니다. 과거에는 규모가 크고 정해진 틀이 있었지만, 최근에는 간소화하거나 실용적인 방향으로 바뀌는 추세입니다.\n\n**3